# Prune4Web Grounder Fine-Tuning — Qwen2.5-0.5B QLoRA

**Goal**: Fine-tune Qwen2.5-0.5B-Instruct as the Action Grounder for Prune4Web using 15,000 training samples from Mind2Web.

**Pipeline**:
1. Install dependencies
2. Stream Mind2Web dataset from HuggingFace (no disk caching — avoids disk-full errors)
3. Generate 15,000 training examples
4. QLoRA fine-tune (4-bit quantized, LoRA r=16, 3 epochs)
5. Evaluate on test_task split (200 samples)
6. Save model, charts, logs to `/kaggle/working/` for download

**GPU**: T4 16GB (Kaggle) — batch_size=4, grad_accum=8, effective batch=32

**Previous results**: 3,690 samples, 2 epochs -> 82.50% Element Accuracy  
**Target**: 15,000 samples, 3 epochs -> 85-88% EA (paper: 88.28%)

**IMPORTANT**: In Kaggle Settings, select **GPU T4 x2** (not P100). Enable **Internet**.

## 1. Install Dependencies

In [ ]:
%%time
!pip install -q \
    transformers>=4.45.0 \
    trl>=0.12.0 \
    peft>=0.13.0 \
    bitsandbytes>=0.44.0 \
    datasets>=3.0.0 \
    accelerate>=1.0.0 \
    matplotlib \
    pandas \
    pyarrow \
    beautifulsoup4 \
    lxml

print("Dependencies installed.")

## 2. Configuration & Setup

In [ ]:
import os
import json
import re
import random
import time
import logging
import sys
import gc
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from collections import Counter

# IMPORTANT: set before torch initializes CUDA; requires kernel restart if changed mid-session.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image

# ── Paths ──
OUTPUT_DIR = Path("/kaggle/working/prune4web_grounder")
MODEL_DIR = OUTPUT_DIR / "model"
DATA_DIR = OUTPUT_DIR / "data"
CHARTS_DIR = OUTPUT_DIR / "charts"
LOGS_DIR = OUTPUT_DIR / "logs"
for d in (MODEL_DIR, DATA_DIR, CHARTS_DIR, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Training config ──
CONFIG = {
    # Data generation
    "max_samples": 500,
    "window_size": 20,
    "eval_frac": 0.1,
    "seed": 42,
    # Training
    "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
    "num_epochs": 3,
    "batch_size": 4,
    "grad_accum": 8,       # effective batch = 32
    "lr": 2e-4,
    "max_seq_len": 1024,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "save_steps": 400,
    "eval_steps": 200,
    "log_steps": 20,
    "warmup_ratio": 0.1,
    # Evaluation
    "eval_num_samples": 200,
    "eval_split": "train",
    "max_new_tokens": 128,
}

# ── Logging ──
LOG_FILE = LOGS_DIR / "full_pipeline.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger("prune4web")

# ── GPU info ──
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"CUDA capability: sm_{props.major}{props.minor}")
    print(f"VRAM: {vram_gb:.1f} GB")
    if props.major < 7:
        print(f"\nWARNING: {gpu_name} (sm_{props.major}{props.minor}) is NOT supported by this PyTorch!")
        print("Switch to 'GPU T4 x2' in Kaggle Settings > Accelerator.")
        raise RuntimeError(f"GPU {gpu_name} not compatible. Need sm_70+. Switch to T4.")
else:
    raise RuntimeError("No GPU detected! Enable GPU in Kaggle notebook settings.")

# Check disk space
disk_stat = os.statvfs("/kaggle/working")
free_gb = disk_stat.f_bavail * disk_stat.f_frsize / 1e9
print(f"Disk free: {free_gb:.1f} GB")

print(f"\nConfig: {json.dumps(CONFIG, indent=2)}")

## 3. Data Structures & Helpers

Self-contained replicas of the project's `ElementNode`, `EvalElementNode`, and data generation helpers.

In [ ]:
# ── System prompt (same as Prune4Web grounder) ──
GROUNDER_SYSTEM = """You are the Action Grounder of Prune4Web.

Given a low-level sub-task and a small ranked list of candidate DOM elements,
identify exactly which element to interact with.

IMPORTANT label equivalences — treat these as identical:
  "Add to Cart" = "Add to Bag" = "Add to Basket" = "Buy" (Apple uses "Add to Bag")
  "Cart" = "Bag" = "Basket"

Return ONLY a JSON object:
{
  "element_uid": <integer UID>,
  "action": "<click|type|select|scroll|hover|check>",
  "value": "<text to input; empty string if not applicable>",
  "confidence": <float 0-1>,
  "reasoning": "<=2 sentences"
}
If no candidate matches, use element_uid: -1.
No markdown fences. No preamble. Pure JSON only.
"""


@dataclass
class ElementNode:
    uid: int
    tag: str
    text: str
    aria_label: str
    placeholder: str
    elem_id: str
    name: str
    elem_class: str
    href: str
    value: str
    input_type: str
    role: str
    title: str

    def to_summary(self) -> str:
        parts = [f"[{self.uid}] <{self.tag}"]
        if self.elem_id:
            parts.append(f' id="{self.elem_id}"')
        if self.input_type:
            parts.append(f' type="{self.input_type}"')
        if self.aria_label:
            parts.append(f' aria-label="{self.aria_label}"')
        if self.placeholder:
            parts.append(f' placeholder="{self.placeholder}"')
        parts.append(">")
        if self.text:
            parts.append(f" {self.text[:80]}")
        return "".join(parts)


@dataclass
class EvalElementNode(ElementNode):
    backend_node_id: str = ""


from bs4 import BeautifulSoup


def _safe_json_loads(raw):
    try:
        return json.loads(raw)
    except Exception:
        return None


def _parse_candidate(raw) -> Optional[dict]:
    """Parse a candidate entry; accepts dict, JSON object string, JSON list string."""
    if isinstance(raw, dict):
        return raw
    if isinstance(raw, str):
        parsed = _safe_json_loads(raw)
        if isinstance(parsed, dict):
            return parsed
        if isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):
            return parsed[0]
        return None
    return None


def _iter_candidate_entries(pool):
    """Yield candidate dicts from various pool formats seen across dataset revisions."""
    if pool is None:
        return
    if isinstance(pool, (list, tuple)):
        iterable = pool
    elif isinstance(pool, dict):
        iterable = [pool]
    elif isinstance(pool, str):
        parsed = _safe_json_loads(pool)
        if isinstance(parsed, list):
            iterable = parsed
        elif isinstance(parsed, dict):
            iterable = [parsed]
        else:
            iterable = []
    else:
        iterable = []

    for raw in iterable:
        cand = _parse_candidate(raw)
        if cand is not None:
            yield cand


def _extract_backend_node_id(cand: dict) -> str:
    """Read backend node id from multiple key aliases."""
    if not isinstance(cand, dict):
        return ""
    aliases = [
        "backend_node_id",
        "backendNodeId",
        "backend-id",
        "node_id",
        "nodeId",
        "id",
    ]
    for key in aliases:
        val = cand.get(key)
        if val is not None and str(val).strip() != "":
            return str(val).strip()

    attrs = cand.get("attributes")
    if isinstance(attrs, dict):
        val = attrs.get("backend_node_id") or attrs.get("backendNodeId")
        if val is not None and str(val).strip() != "":
            return str(val).strip()
    return ""


def _extract_html(row) -> str:
    """Get HTML payload using fallback keys used by different dataset snapshots."""
    html = (
        row.get("cleaned_html")
        or row.get("cleaned_html_str")
        or row.get("html")
        or row.get("page_html")
        or row.get("raw_html")
        or row.get("document")
        or ""
    )
    if isinstance(html, bytes):
        try:
            html = html.decode("utf-8", errors="ignore")
        except Exception:
            html = ""
    return str(html)


def _get_candidate_pool(row, positive=True):
    if positive:
        keys = ["pos_candidates", "positive_candidates", "pos", "positive"]
    else:
        keys = ["neg_candidates", "negative_candidates", "neg", "negative"]
    for k in keys:
        if k in row and row.get(k) is not None:
            return row.get(k)
    return None


def _iter_action_samples(row):
    """Normalize trajectory-level rows into action-level rows."""
    actions = row.get("actions")
    if not isinstance(actions, list) or len(actions) == 0:
        yield row
        return

    action_reprs = row.get("action_reprs", [])
    ann_id = str(row.get("annotation_id", "ann"))
    task = str(row.get("confirmed_task", row.get("task", "")))

    for idx, action in enumerate(actions):
        if not isinstance(action, dict):
            continue
        sample = dict(action)
        sample.setdefault("confirmed_task", task)
        sample.setdefault("action_uid", f"{ann_id}_{idx}")

        if "target_action_reprs" not in sample:
            if isinstance(action_reprs, list) and idx < len(action_reprs):
                sample["target_action_reprs"] = action_reprs[idx]
            elif isinstance(action_reprs, str):
                sample["target_action_reprs"] = action_reprs
            else:
                sample["target_action_reprs"] = ""

        if not _extract_html(sample):
            sample["cleaned_html"] = _extract_html(row)

        if _get_candidate_pool(sample, positive=True) is None:
            sample["pos_candidates"] = _get_candidate_pool(row, positive=True)
        if _get_candidate_pool(sample, positive=False) is None:
            sample["neg_candidates"] = _get_candidate_pool(row, positive=False)

        yield sample


def parse_candidates_from_pool(html: str, pos_candidates, neg_candidates) -> List[EvalElementNode]:
    if not html:
        return []
    soup = BeautifulSoup(html, "lxml")
    bnode_map = {}
    for el in soup.find_all(attrs={"backend_node_id": True}):
        bnode_map[str(el.get("backend_node_id"))] = el

    all_cand_ids: List[str] = []
    for pool in (pos_candidates, neg_candidates):
        for cand in _iter_candidate_entries(pool):
            bid = _extract_backend_node_id(cand)
            if bid:
                all_cand_ids.append(bid)

    nodes: List[EvalElementNode] = []
    uid = 0
    seen: set = set()
    for node_id in all_cand_ids:
        if not node_id or node_id in seen:
            continue
        seen.add(node_id)

        el = bnode_map.get(node_id)
        if el is None:
            nodes.append(EvalElementNode(
                uid=uid, tag="", text="", aria_label="", placeholder="",
                elem_id="", name="", elem_class="", href="", value="",
                input_type="", role="", title="", backend_node_id=node_id,
            ))
            uid += 1
            continue

        tag_name = (el.name or "").lower()
        role = str(el.get("role", "")).lower()
        cls = el.get("class", [])
        cls_str = cls if isinstance(cls, str) else " ".join(cls)

        nodes.append(EvalElementNode(
            uid=uid,
            tag=tag_name,
            text=el.get_text(separator=" ", strip=True)[:200],
            aria_label=el.get("aria-label", el.get("aria_label", "")),
            placeholder=el.get("placeholder", ""),
            elem_id=el.get("id", ""),
            name=el.get("name", ""),
            elem_class=cls_str,
            href=el.get("href", ""),
            value=el.get("value", ""),
            input_type=el.get("type", ""),
            role=role,
            title=el.get("title", ""),
            backend_node_id=node_id,
        ))
        uid += 1

    return nodes


def get_gt_backend_node_id(row) -> Optional[str]:
    pos = _get_candidate_pool(row, positive=True)
    for cand in _iter_candidate_entries(pos):
        bid = _extract_backend_node_id(cand)
        if bid:
            return bid
    return None


def get_gt_action_type(row) -> str:
    op = row.get("operation", row.get("action", "{}"))
    if isinstance(op, str):
        parsed = _safe_json_loads(op)
        op = parsed if parsed is not None else {"action": op}

    if not isinstance(op, dict):
        return "click"

    op_type = str(
        op.get("op", op.get("original_op", op.get("action", op.get("action_type", "CLICK"))))
    ).upper()
    mapping = {
        "CLICK": "click",
        "TYPE": "type",
        "SELECT": "select",
        "SCROLL": "scroll",
        "HOVER": "hover",
        "CHECK": "check",
    }
    return mapping.get(op_type, "click")


def parse_target_action_repr(target_repr: str) -> Tuple[str, str]:
    if not target_repr:
        return "", "click"
    parts = target_repr.rsplit(" -> ", 1)
    action = parts[1].strip().lower() if len(parts) > 1 else "click"
    desc = parts[0].strip()
    desc = re.sub(r"^\[.*?\]\s*", "", desc).strip()
    return desc, action


def build_candidate_window(
    all_elements: List[EvalElementNode],
    gt_node_id: str,
    window_size: int = 20,
    rng: Optional[random.Random] = None,
) -> Tuple[List[EvalElementNode], int]:
    rng = rng or random.Random()
    gt_el = next((e for e in all_elements if e.backend_node_id == gt_node_id), None)
    if gt_el is None:
        return [], -1
    neg_pool = [e for e in all_elements if e.backend_node_id != gt_node_id]
    negs = neg_pool if len(neg_pool) <= window_size - 1 else rng.sample(neg_pool, window_size - 1)
    window = [gt_el] + negs
    rng.shuffle(window)
    for i, el in enumerate(window):
        el.uid = i
    gt_index = next(i for i, el in enumerate(window) if el.backend_node_id == gt_node_id)
    return window, gt_index


def format_user_message(sub_task: str, candidates: List[EvalElementNode], value_hint: str = "") -> str:
    candidate_lines = [f"  {i + 1}. {el.to_summary()}" for i, el in enumerate(candidates)]
    return (
        f"Sub-task: {sub_task}\nValue hint: {value_hint}\n\n"
        f"Candidate elements (ranked by relevance):\n" + "\n".join(candidate_lines)
    )


def format_assistant_message(gt_uid: int, action: str, value: str, element_desc: str) -> str:
    payload = {
        "element_uid": int(gt_uid),
        "action": action,
        "value": value,
        "confidence": 0.95,
        "reasoning": f"Element {gt_uid} matches the sub-task target: {element_desc[:80]}",
    }
    return json.dumps(payload, ensure_ascii=False)


def value_from_operation(operation_raw) -> str:
    if operation_raw is None:
        return ""
    try:
        op = json.loads(operation_raw) if isinstance(operation_raw, str) else operation_raw
    except Exception:
        return ""
    if isinstance(op, str):
        return op.strip()
    if not isinstance(op, dict):
        return ""
    return str(op.get("value") or op.get("action_input") or op.get("text") or "").strip()


# ── Quick sanity check: stream 1 row and test parsing ──
from datasets import load_dataset as hf_load_dataset
_test_ds = hf_load_dataset("osunlp/Mind2Web", split="train", streaming=True, trust_remote_code=True)
_test_row = next(iter(_test_ds))

print("Top-level row keys:", sorted(list(_test_row.keys())))
_samples = list(_iter_action_samples(_test_row))
print(f"action-level samples in first row: {len(_samples)}")
if _samples:
    _s0 = _samples[0]
    print("sample keys:", sorted(list(_s0.keys()))[:25], "...")
    _pc = _get_candidate_pool(_s0, positive=True)
    _nc = _get_candidate_pool(_s0, positive=False)
    print(f"pos_candidates type: {type(_pc)}, len: {len(_pc) if hasattr(_pc, '__len__') else 'NA'}")
    print(f"neg_candidates type: {type(_nc)}, len: {len(_nc) if hasattr(_nc, '__len__') else 'NA'}")

    _gt = get_gt_backend_node_id(_s0)
    _html = _extract_html(_s0)
    print(f"gt_backend_node_id: {_gt}")
    print(f"html len: {len(_html)}")

    _elems = parse_candidates_from_pool(_html, _pc, _nc)
    print(f"parsed candidates: {len(_elems)}")

del _test_ds, _test_row
print("\nData structures and helpers ready.")

## 4. Generate Training Data (15,000 samples)

Uses **streaming** mode to avoid downloading the entire dataset to disk. Only text columns are read (no screenshots), and rows are processed on-the-fly.

In [ ]:
%%time
from datasets import load_dataset as hf_load_dataset

def row_to_training_example(row, rng, window_size=20):
    """Convert an action-level row to a chat-formatted training example with skip reason."""
    gt_node_id = get_gt_backend_node_id(row)
    if not gt_node_id:
        return None, "missing_gt_backend_node_id"

    html = _extract_html(row)
    if not html or len(html) < 100:
        return None, "missing_or_short_html"

    pos_pool = _get_candidate_pool(row, positive=True)
    neg_pool = _get_candidate_pool(row, positive=False)
    all_elements = parse_candidates_from_pool(html, pos_pool, neg_pool)
    if not all_elements:
        return None, "no_parsed_candidates"

    window, gt_index = build_candidate_window(all_elements, gt_node_id, window_size, rng)
    if not window or gt_index < 0:
        return None, "gt_not_in_window"

    target_repr = str(row.get("target_action_reprs", row.get("action_repr", "")))
    element_desc, repr_action = parse_target_action_repr(target_repr)
    sub_task = element_desc or str(row.get("confirmed_task", row.get("task", "")))
    if not sub_task.strip():
        sub_task = "Interact with the correct element for the current step."

    action = get_gt_action_type(row) or repr_action or "click"
    value = value_from_operation(row.get("operation", row.get("action")))

    user_msg = format_user_message(sub_task, window, value)
    assistant_msg = format_assistant_message(gt_index, action, value, element_desc)

    return {
        "messages": [
            {"role": "system", "content": GROUNDER_SYSTEM},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ],
        "_meta": {
            "action_uid": str(row.get("action_uid", "")),
            "gt_backend_node_id": gt_node_id,
            "gt_index": gt_index,
            "action": action,
            "window_size": len(window),
            "dom_size": len(all_elements),
        },
    }, "ok"


# ── Stream Mind2Web train split (no disk caching!) ──
log.info("Streaming Mind2Web train split from HuggingFace...")
log.info(f"Target: {CONFIG['max_samples']} samples, window_size={CONFIG['window_size']}")

train_stream = hf_load_dataset(
    "osunlp/Mind2Web",
    split="train",
    streaming=True,
    trust_remote_code=True,
)
# Shuffle with a buffer for randomness
train_stream = train_stream.shuffle(seed=CONFIG["seed"], buffer_size=2000)

rng = random.Random(CONFIG["seed"])
examples = []
skipped = 0
skip_reasons = Counter()
t0 = time.time()

for traj_row in train_stream:
    for row in _iter_action_samples(traj_row):
        if len(examples) >= CONFIG["max_samples"]:
            break
        try:
            ex, reason = row_to_training_example(row, rng, CONFIG["window_size"])
        except Exception as e:
            skipped += 1
            skip_reasons[f"exception:{type(e).__name__}"] += 1
            continue

        if ex is None:
            skipped += 1
            skip_reasons[reason] += 1
            continue

        examples.append(ex)
        if len(examples) % 2000 == 0:
            log.info(f"  generated {len(examples)}/{CONFIG['max_samples']} (skipped {skipped})")

    if len(examples) >= CONFIG["max_samples"]:
        break

elapsed_gen = time.time() - t0
log.info(f"Generated {len(examples)} examples (skipped {skipped}) in {elapsed_gen:.1f}s")

if skip_reasons:
    log.info(f"Top skip reasons: {dict(skip_reasons.most_common(8))}")

if len(examples) == 0:
    print(f"\n{'='*60}")
    print("DATA GENERATION SUMMARY")
    print(f"{'='*60}")
    print("Total examples : 0")
    print("Train / Eval   : 0 / 0")
    print(f"Skipped rows   : {skipped}")
    print(f"Time           : {elapsed_gen:.1f}s")
    print(f"Skip reasons   : {dict(skip_reasons.most_common(12))}")
    print(f"{'='*60}")

    # Probe one row for debugging so failures are actionable in Kaggle logs.
    _probe = hf_load_dataset(
        "osunlp/Mind2Web",
        split="train",
        streaming=True,
        trust_remote_code=True,
    )
    _probe_row = next(iter(_probe))
    _probe_samples = list(_iter_action_samples(_probe_row))
    log.error(f"Probe top-level row keys: {list(_probe_row.keys())}")
    log.error(f"Probe action sample count: {len(_probe_samples)}")
    if _probe_samples:
        s0 = _probe_samples[0]
        log.error(f"Probe action keys: {list(s0.keys())}")
        log.error(
            "Probe field types: "
            f"pos={type(_get_candidate_pool(s0, True))}, "
            f"neg={type(_get_candidate_pool(s0, False))}, "
            f"html_len={len(_extract_html(s0))}"
        )
    raise RuntimeError(
        "No training examples generated. Check skip reasons above; dataset field format may differ "
        "from parser assumptions."
    )

# ── Train/eval split ──
rng.shuffle(examples)
n_eval = max(1, int(len(examples) * CONFIG["eval_frac"]))
eval_set = examples[:n_eval]
train_set = examples[n_eval:]
log.info(f"Split: train={len(train_set)}, eval={len(eval_set)}")

# ── Write JSONL files ──
TRAIN_FILE = DATA_DIR / "train.jsonl"
EVAL_FILE = DATA_DIR / "eval.jsonl"

for path, items in [(TRAIN_FILE, train_set), (EVAL_FILE, eval_set)]:
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            out = {"messages": it["messages"], "meta": it.get("_meta", {})}
            f.write(json.dumps(out, ensure_ascii=False) + "\n")

log.info(f"Written: {TRAIN_FILE} ({len(train_set)} examples)")
log.info(f"Written: {EVAL_FILE} ({len(eval_set)} examples)")

# ── Data stats ──
gt_positions = [ex["_meta"]["gt_index"] for ex in examples]
actions = [ex["_meta"]["action"] for ex in examples]
act_counter = Counter(actions)

print(f"\n{'='*60}")
print(f"DATA GENERATION SUMMARY")
print(f"{'='*60}")
print(f"Total examples : {len(examples)}")
print(f"Train / Eval   : {len(train_set)} / {len(eval_set)}")
print(f"Skipped rows   : {skipped}")
print(f"Time           : {elapsed_gen:.1f}s")
print(f"GT-index dist  : min={min(gt_positions)} max={max(gt_positions)} mean={sum(gt_positions)/len(gt_positions):.1f}")
print(f"Action dist    : {dict(act_counter)}")
print(f"{'='*60}")

In [ ]:
# ── Visualize data distribution ──
import math

# Rebuild stats from in-memory examples when available; otherwise load from saved JSONL.
if "examples" in globals() and isinstance(examples, list) and len(examples) > 0:
    _examples_for_stats = examples
else:
    _examples_for_stats = []
    for _path in [TRAIN_FILE, EVAL_FILE]:
        if _path.exists() and _path.stat().st_size > 0:
            with open(_path, "r", encoding="utf-8") as _f:
                for _line in _f:
                    try:
                        _row = json.loads(_line)
                        _meta = _row.get("meta", {})
                        if isinstance(_meta, dict):
                            _examples_for_stats.append({"_meta": _meta})
                    except Exception:
                        continue

if not _examples_for_stats:
    raise RuntimeError(
        "No examples available for plotting. Run data generation cell first and ensure train/eval JSONL are written."
    )

gt_positions = [ex["_meta"]["gt_index"] for ex in _examples_for_stats if "gt_index" in ex.get("_meta", {})]
actions = [ex["_meta"]["action"] for ex in _examples_for_stats if "action" in ex.get("_meta", {})]
dom_sizes = [ex["_meta"]["dom_size"] for ex in _examples_for_stats if "dom_size" in ex.get("_meta", {})]

if not gt_positions or not actions or not dom_sizes:
    raise RuntimeError(
        "Insufficient metadata for plotting. Expected gt_index/action/dom_size in JSONL meta fields."
    )

act_counter = Counter(actions)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. GT position distribution
axes[0].hist(gt_positions, bins=CONFIG["window_size"], color="#1f77b4", edgecolor="white", alpha=0.85)
axes[0].set_xlabel("GT element index in window")
axes[0].set_ylabel("Count")
axes[0].set_title("Ground Truth Position Distribution")
axes[0].axhline(len(gt_positions) / CONFIG["window_size"], color="red", linestyle="--", alpha=0.5, label="Uniform")
axes[0].legend()

# 2. Action type distribution
act_labels = list(act_counter.keys())
act_values = list(act_counter.values())
colors = ["#2ca02c", "#ff7f0e", "#d62728", "#9467bd", "#8c564b", "#e377c2"]
axes[1].bar(act_labels, act_values, color=colors[:len(act_labels)])
axes[1].set_xlabel("Action type")
axes[1].set_ylabel("Count")
axes[1].set_title("Action Type Distribution")
for i, (l, v) in enumerate(zip(act_labels, act_values)):
    axes[1].text(i, v + max(1, math.ceil(max(act_values) * 0.02)), str(v), ha="center", fontsize=9)

# 3. DOM size distribution
axes[2].hist(dom_sizes, bins=50, color="#ff7f0e", edgecolor="white", alpha=0.85)
axes[2].set_xlabel("DOM size (candidates in pool)")
axes[2].set_ylabel("Count")
axes[2].set_title("Candidate Pool Size Distribution")

fig.suptitle(f"Training Data Statistics — {len(_examples_for_stats)} samples", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(CHARTS_DIR / "data_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
log.info(f"Data distribution chart saved to {CHARTS_DIR / 'data_distribution.png'}")

# Free only if present
if "examples" in globals():
    del examples
if "train_stream" in globals():
    del train_stream
gc.collect()

## 5. QLoRA Fine-Tuning

In [ ]:
%%time
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
 )
from trl import SFTConfig, SFTTrainer

log.info("=" * 72)
log.info("STARTING FINE-TUNING")
log.info("=" * 72)

# ── Guard: fail early if data files are empty ──
if (not TRAIN_FILE.exists()) or (not EVAL_FILE.exists()):
    raise RuntimeError(
        f"Training files not found: TRAIN_FILE={TRAIN_FILE.exists()}, EVAL_FILE={EVAL_FILE.exists()}"
    )
if TRAIN_FILE.stat().st_size == 0 or EVAL_FILE.stat().st_size == 0:
    raise RuntimeError(
        "Training/eval JSONL is empty. Re-run data generation and inspect skip reasons before fine-tuning."
    )

# ── Load tokenizer ──
log.info("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ── Force model on a single GPU to avoid DataParallel cross-device mismatch ──
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook.")
torch.cuda.set_device(0)
single_gpu_device_map = {"": 0}
log.info(f"Visible CUDA devices: {torch.cuda.device_count()} | Using device_map={single_gpu_device_map}")

# ── Load model in 4-bit (T4-safe fp16 compute) ──
log.info("Loading base model in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
 )
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    quantization_config=bnb_config,
    device_map=single_gpu_device_map,
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

log.info(f"Base params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
log.info(f"VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── LoRA config ──
lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
log.info(f"Trainable: {trainable/1e6:.2f}M ({100*trainable/total:.2f}%)")
log.info(f"VRAM after LoRA: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Load training data ──
log.info("Loading training JSONL...")
ds = load_dataset(
    "json",
    data_files={"train": str(TRAIN_FILE), "eval": str(EVAL_FILE)},
)
for split in ("train", "eval"):
    if "meta" in ds[split].column_names:
        ds[split] = ds[split].remove_columns("meta")
log.info(f"Train: {len(ds['train'])} | Eval: {len(ds['eval'])}")

est_steps = len(ds['train']) // (CONFIG['batch_size'] * CONFIG['grad_accum']) * CONFIG['num_epochs']
print(f"\nReady to train. Estimated steps: {est_steps}")

In [ ]:
%%time

# ── Loss tracker callback ──
class LossTrackerCallback(TrainerCallback):
    def __init__(self):
        self.train_steps, self.train_losses = [], []
        self.eval_steps, self.eval_losses = [], []
        self.lr_steps, self.lr_values = [], []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step
        if "loss" in logs and "eval_loss" not in logs:
            self.train_steps.append(step)
            self.train_losses.append(float(logs["loss"]))
            if "learning_rate" in logs:
                self.lr_steps.append(step)
                self.lr_values.append(float(logs["learning_rate"]))
            log.info(f"  step {step:>5}  train_loss={logs['loss']:.4f}"
                     + (f"  lr={logs.get('learning_rate', 0):.2e}" if 'learning_rate' in logs else ""))
        if "eval_loss" in logs:
            self.eval_steps.append(step)
            self.eval_losses.append(float(logs["eval_loss"]))
            log.info(f"  step {step:>5}  eval_loss={logs['eval_loss']:.4f}")


class PeriodicStateSaver(TrainerCallback):
    """Saves loss tracker data to disk every N steps for crash recovery."""
    def __init__(self, loss_cb, save_every=100):
        self.loss_cb = loss_cb
        self.save_every = save_every

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.global_step % self.save_every == 0 and state.global_step > 0:
            data = {
                "train_steps": self.loss_cb.train_steps,
                "train_losses": self.loss_cb.train_losses,
                "eval_steps": self.loss_cb.eval_steps,
                "eval_losses": self.loss_cb.eval_losses,
                "lr_steps": self.loss_cb.lr_steps,
                "lr_values": self.loss_cb.lr_values,
            }
            with open(LOGS_DIR / "loss_tracker_checkpoint.json", "w") as f:
                json.dump(data, f)


# ── SFT config ──
sft_config = SFTConfig(
    output_dir=str(MODEL_DIR),
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    learning_rate=CONFIG["lr"],
    lr_scheduler_type="cosine",
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=CONFIG["log_steps"],
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["eval_steps"],  # must align with eval_steps for load_best_model_at_end
    save_total_limit=2,
    bf16=False,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    max_length=CONFIG["max_seq_len"],
    dataset_num_proc=2,
    report_to="none",
    seed=CONFIG["seed"],
    data_seed=CONFIG["seed"],
    remove_unused_columns=False,
    disable_tqdm=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# ── Trainer ──
loss_cb = LossTrackerCallback()
state_saver = PeriodicStateSaver(loss_cb, save_every=100)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    eval_dataset=ds["eval"],
    processing_class=tokenizer,
    callbacks=[loss_cb, state_saver],
)

# ── TRAIN ──
log.info("Starting training...")
log.info(f"  VRAM before train: {torch.cuda.memory_allocated()/1e9:.2f} GB")

t_train_start = time.time()
train_result = trainer.train()
t_train_elapsed = time.time() - t_train_start

log.info(f"Training finished in {t_train_elapsed/60:.1f} min")
log.info(f"Train result: {train_result.metrics}")

# ── Final eval ──
log.info("Running final evaluation...")
eval_metrics = trainer.evaluate()
log.info(f"Final eval metrics: {eval_metrics}")

# ── Save adapter + tokenizer ──
FINAL_MODEL_DIR = OUTPUT_DIR / "final_adapter"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
log.info(f"Saving adapter to {FINAL_MODEL_DIR}")
trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

# ── Save training metadata ──
training_meta = {
    "base_model": CONFIG["base_model"],
    "num_train_examples": len(ds["train"]),
    "num_eval_examples": len(ds["eval"]),
    "num_epochs": CONFIG["num_epochs"],
    "batch_size": CONFIG["batch_size"],
    "grad_accum": CONFIG["grad_accum"],
    "effective_batch": CONFIG["batch_size"] * CONFIG["grad_accum"],
    "lr": CONFIG["lr"],
    "max_seq_len": CONFIG["max_seq_len"],
    "lora_r": CONFIG["lora_r"],
    "lora_alpha": CONFIG["lora_alpha"],
    "final_eval_loss": eval_metrics.get("eval_loss"),
    "train_runtime_min": round(t_train_elapsed / 60, 2),
    "train_metrics": train_result.metrics,
    "eval_metrics": eval_metrics,
}
with open(FINAL_MODEL_DIR / "training_meta.json", "w", encoding="utf-8") as f:
    json.dump(training_meta, f, indent=2)

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Duration       : {t_train_elapsed/60:.1f} min")
print(f"Final eval loss: {eval_metrics.get('eval_loss', 'N/A'):.4f}")
print(f"Adapter saved  : {FINAL_MODEL_DIR}")
print(f"{'='*60}")

## 6. Training Charts

In [ ]:
# ── Plot loss curves ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

ax = axes[0]
if loss_cb.train_steps:
    ax.plot(loss_cb.train_steps, loss_cb.train_losses, label="Train loss",
            color="#1f77b4", linewidth=1.2, alpha=0.7, marker="o", markersize=2)
if loss_cb.eval_steps:
    ax.plot(loss_cb.eval_steps, loss_cb.eval_losses, label="Eval loss",
            color="#d62728", linewidth=2.0, marker="s", markersize=5)
ax.set_xlabel("Training step")
ax.set_ylabel("Loss")
ax.set_title("Prune4Web Grounder — Loss Curve (Qwen2.5-0.5B QLoRA)")
ax.grid(alpha=0.3)
ax.legend()

ax = axes[1]
if loss_cb.lr_steps:
    ax.plot(loss_cb.lr_steps, loss_cb.lr_values, color="#2ca02c", linewidth=1.5)
ax.set_xlabel("Training step")
ax.set_ylabel("Learning rate")
ax.set_title("Learning Rate Schedule (Cosine)")
ax.grid(alpha=0.3)
ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))

fig.tight_layout()
fig.savefig(CHARTS_DIR / "loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()
log.info(f"Loss curve saved to {CHARTS_DIR / 'loss_curve.png'}")

# Save raw loss data
loss_data = {
    "train_steps": loss_cb.train_steps,
    "train_losses": loss_cb.train_losses,
    "eval_steps": loss_cb.eval_steps,
    "eval_losses": loss_cb.eval_losses,
    "lr_steps": loss_cb.lr_steps,
    "lr_values": loss_cb.lr_values,
}
with open(LOGS_DIR / "loss_data.json", "w") as f:
    json.dump(loss_data, f)
print(f"Raw loss data saved to {LOGS_DIR / 'loss_data.json'}")

## 7. Evaluation on test_task (200 samples)

In [ ]:
%%time
from peft import PeftModel

# ── Free training memory ──
del trainer, model
torch.cuda.empty_cache()
gc.collect()
log.info(f"VRAM freed. Now: {torch.cuda.memory_allocated()/1e9:.2f} GB used")

# ── Load fresh base model + adapter for eval on single GPU ──
torch.cuda.set_device(0)
single_gpu_device_map = {"": 0}
log.info("Loading base model for evaluation...")
eval_tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])
if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

eval_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    ),
    device_map=single_gpu_device_map,
    torch_dtype=torch.float16,
    attn_implementation="eager",
)

log.info(f"Loading LoRA adapter from {FINAL_MODEL_DIR}")
eval_model = PeftModel.from_pretrained(eval_model, str(FINAL_MODEL_DIR))
eval_model.eval()
log.info(f"VRAM after eval model load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
def extract_json(text: str) -> Optional[Dict]:
    """Extract a JSON object from model output."""
    if not text:
        return None
    match = re.search(r'\{[^{}]*?"element_uid"[^{}]*\}', text, re.DOTALL)
    if not match:
        match = re.search(r'\{[\s\S]*\}', text)
    if not match:
        return None
    raw = match.group(0)
    try:
        return json.loads(raw)
    except Exception:
        try:
            return json.loads(raw.rsplit("}", 1)[0] + "}")
        except Exception:
            return None

print("JSON extractor ready.")

In [ ]:
%%time

# ── Stream test data (same approach — no disk caching) ──
from datasets import get_dataset_split_names

requested_split = CONFIG["eval_split"]
try:
    available_splits = get_dataset_split_names("osunlp/Mind2Web", trust_remote_code=True)
except Exception as e:
    log.warning(f"Could not fetch split names ({type(e).__name__}); defaulting to ['train']")
    available_splits = ["train"]

if requested_split in available_splits:
    eval_split_to_use = requested_split
else:
    eval_split_to_use = "train" if "train" in available_splits else available_splits[0]
    log.warning(
        f"Requested eval split '{requested_split}' is unavailable. "
        f"Using '{eval_split_to_use}' from available splits: {available_splits}"
    )

log.info(f"Streaming eval split '{eval_split_to_use}' from HuggingFace...")
test_stream = hf_load_dataset(
    "osunlp/Mind2Web",
    split=eval_split_to_use,
    streaming=True,
    trust_remote_code=True,
)
test_stream = test_stream.shuffle(seed=CONFIG["seed"], buffer_size=2000)

rng_eval = random.Random(CONFIG["seed"])

# ── Run evaluation ──
results = []
format_ok = 0
element_correct = 0
op_correct = 0
by_action_total = Counter()
by_action_correct = Counter()
skipped_eval = 0

t_eval_start = time.time()
target_n = CONFIG["eval_num_samples"]

for traj_row in test_stream:
    for row in _iter_action_samples(traj_row):
        if len(results) >= target_n:
            break

        gt_node_id = get_gt_backend_node_id(row)
        if not gt_node_id:
            skipped_eval += 1
            continue

        html = _extract_html(row)
        if not html or len(str(html)) < 100:
            skipped_eval += 1
            continue

        pos_pool = _get_candidate_pool(row, positive=True)
        neg_pool = _get_candidate_pool(row, positive=False)
        all_elements = parse_candidates_from_pool(html, pos_pool, neg_pool)
        if not all_elements:
            skipped_eval += 1
            continue

        window, gt_index = build_candidate_window(all_elements, gt_node_id, CONFIG["window_size"], rng_eval)
        if not window or gt_index < 0:
            skipped_eval += 1
            continue

        target_repr = str(row.get("target_action_reprs", row.get("action_repr", "")))
        element_desc, _ = parse_target_action_repr(target_repr)
        sub_task = element_desc or str(row.get("confirmed_task", row.get("task", "")))
        gt_action = get_gt_action_type(row)
        value_hint = value_from_operation(row.get("operation", row.get("action")))

        user_msg = format_user_message(sub_task, window, value_hint)
        messages = [
            {"role": "system", "content": GROUNDER_SYSTEM},
            {"role": "user", "content": user_msg},
        ]

        prompt_text = eval_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = eval_tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=1024).to(eval_model.device)

        with torch.no_grad():
            out = eval_model.generate(
                **inputs,
                max_new_tokens=CONFIG["max_new_tokens"],
                do_sample=False,
                temperature=1.0,
                top_p=1.0,
                pad_token_id=eval_tokenizer.pad_token_id,
            )

        gen_text = eval_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        parsed = extract_json(gen_text)

        pred_uid = -1
        pred_action = ""
        if parsed is not None:
            format_ok += 1
            try:
                pred_uid = int(parsed.get("element_uid", -1))
            except Exception:
                pred_uid = -1
            pred_action = str(parsed.get("action", "")).lower()

        el_correct = (pred_uid == gt_index)
        op_correct_here = el_correct and (pred_action == gt_action)
        by_action_total[gt_action] += 1
        if el_correct:
            element_correct += 1
            by_action_correct[gt_action] += 1
        if op_correct_here:
            op_correct += 1

        results.append({
            "action_uid": str(row.get("action_uid", "")),
            "sub_task": sub_task[:100],
            "gt_index": gt_index,
            "pred_uid": pred_uid,
            "gt_action": gt_action,
            "pred_action": pred_action,
            "el_correct": el_correct,
            "op_correct": op_correct_here,
            "format_ok": parsed is not None,
            "raw_output": gen_text[:200],
        })

        mark = "OK" if el_correct else "X"
        if len(results) % 20 == 0 or len(results) <= 5:
            log.info(f"[{len(results):>4}/{target_n}] gt={gt_index} pred={pred_uid} "
                     f"gt_act={gt_action} pred_act={pred_action} [{mark}]")

    if len(results) >= target_n:
        break

t_eval_elapsed = time.time() - t_eval_start
n = len(results)

ea = element_correct / n * 100 if n else 0
opa = op_correct / n * 100 if n else 0
fmt = format_ok / n * 100 if n else 0

print(f"\n{'='*72}")
print(f"EVALUATION RESULTS")
print(f"{'='*72}")
print(f"Samples evaluated : {n} (skipped {skipped_eval})")
print(f"Elapsed           : {t_eval_elapsed:.1f}s ({t_eval_elapsed/max(n,1):.2f}s/sample)")
print(f"Format validity   : {fmt:.2f}%")
print(f"Element Accuracy  : {ea:.2f}%   (paper: 88.28% | prev run: 82.50% | gpt-4o-mini: 72.22%)")
print(f"Op Accuracy       : {opa:.2f}%")
print()
print("Element Accuracy by action type:")
for act, total in by_action_total.most_common():
    correct = by_action_correct[act]
    pct = correct / total * 100 if total else 0
    print(f"  {act:<8}: {correct}/{total} ({pct:.1f}%)")
print(f"{'='*72}")

log.info(f"Element Accuracy: {ea:.2f}%")
log.info(f"Op Accuracy: {opa:.2f}%")

## 8. Evaluation Charts

In [ ]:
# ── Chart 1: Headline metrics + per-action breakdown ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

ax = axes[0]
metric_names = ["Format\nValidity", "Element\nAccuracy", "Op\nAccuracy"]
ours = [fmt, ea, opa]
bar_colors = ["#2ca02c", "#1f77b4", "#ff7f0e"]
bars = ax.bar(metric_names, ours, color=bar_colors, width=0.55)
ax.axhline(88.28, color="#d62728", linestyle="--", linewidth=1.5, label="Paper EA: 88.28%")
ax.axhline(82.50, color="#9467bd", linestyle="-.", linewidth=1.5, label="Prev run EA: 82.50%")
ax.axhline(72.22, color="#7f7f7f", linestyle=":", linewidth=1.5, label="GPT-4o-mini: 72.22%")
ax.set_ylim(0, 105)
ax.set_ylabel("Percentage")
ax.set_title(f"Prune4Web Grounder — Fine-tuned Qwen2.5-0.5B ({n} samples, {CONFIG['eval_split']})")
for b, v in zip(bars, ours):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1.5, f"{v:.1f}%",
            ha="center", fontsize=12, weight="bold")
ax.legend(loc="lower right", fontsize=9)

ax = axes[1]
actions_list = list(by_action_total.keys())
accs = [by_action_correct[a] / by_action_total[a] * 100 if by_action_total[a] else 0 for a in actions_list]
totals = [by_action_total[a] for a in actions_list]
bars2 = ax.bar(actions_list, accs, color="#1f77b4", width=0.55)
ax.set_ylabel("Element Accuracy (%)")
ax.set_ylim(0, 105)
ax.set_title("Element Accuracy by Action Type")
for b, v, t in zip(bars2, accs, totals):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1.5,
            f"{v:.1f}%\n(n={t})", ha="center", fontsize=10)

fig.tight_layout()
fig.savefig(CHARTS_DIR / "eval_results.png", dpi=150, bbox_inches="tight")
plt.show()
log.info(f"Eval charts saved to {CHARTS_DIR / 'eval_results.png'}")

In [ ]:
# ── Chart 2: Comparison bar chart ──
fig, ax = plt.subplots(figsize=(10, 5.5))

# Safely get training count
try:
    _train_count = len(train_set)
except NameError:
    _train_count = sum(1 for _ in open(TRAIN_FILE))

models = ["GPT-4o-mini
(baseline)", "Prev run
(3.7K, 2ep)", f"This run
({_train_count/1000:.0f}K, {CONFIG['num_epochs']}ep)", "Paper
(full Mind2Web)"]
ea_values = [72.22, 82.50, ea, 88.28]
colors = ["#7f7f7f", "#9467bd", "#1f77b4", "#d62728"]

bars = ax.bar(models, ea_values, color=colors, width=0.55, edgecolor="white", linewidth=1.5)
ax.set_ylabel("Element Accuracy (%)")
ax.set_ylim(0, 105)
ax.set_title("Prune4Web Grounder — Element Accuracy Comparison")

for b, v in zip(bars, ea_values):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1.5, f"{v:.2f}%",
            ha="center", fontsize=12, weight="bold")

ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "comparison.png", dpi=150, bbox_inches="tight")
plt.show()
log.info(f"Comparison chart saved to {CHARTS_DIR / 'comparison.png'}")

## 9. Save All Results

In [ ]:
# ── Save evaluation results JSON ──
try:
    _train_count = len(train_set)
except NameError:
    _train_count = sum(1 for _ in open(TRAIN_FILE))

eval_results_path = DATA_DIR / "eval_results.json"
eval_output = {
    "config": {
        "split": CONFIG["eval_split"],
        "num_samples": n,
        "window_size": CONFIG["window_size"],
        "base_model": CONFIG["base_model"],
        "adapter_dir": str(FINAL_MODEL_DIR),
        "training_samples": _train_count,
        "num_epochs": CONFIG["num_epochs"],
    },
    "metrics": {
        "element_accuracy": ea,
        "op_accuracy": opa,
        "format_validity": fmt,
        "paper_element_accuracy": 88.28,
        "prev_run_element_accuracy": 82.50,
        "gpt_4o_mini_baseline": 72.22,
    },
    "by_action": {
        act: {"total": by_action_total[act], "correct": by_action_correct[act]}
        for act in by_action_total
    },
    "training_meta": training_meta,
    "results": results,
}
with open(eval_results_path, "w", encoding="utf-8") as f:
    json.dump(eval_output, f, indent=2)
log.info(f"Eval results saved to {eval_results_path}")

# ── Summary of all output files ──
print(f"
{'='*72}")
print(f"ALL OUTPUT FILES (in /kaggle/working/prune4web_grounder/)")
print(f"{'='*72}")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(str(OUTPUT_DIR), "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files:
        fpath = os.path.join(root, file)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"{subindent}{file} ({size_mb:.2f} MB)")
print(f"{'='*72}")

print(f"
Done! Download 'final_adapter' folder from /kaggle/working/prune4web_grounder/")
print(f"To use locally: copy final_adapter/ to your model directory.")

## 10. Quick Sanity Check — Sample Predictions

In [ ]:
# Show sample predictions (mix of correct and incorrect)
correct_samples = [r for r in results if r["el_correct"]]
incorrect_samples = [r for r in results if not r["el_correct"]]

print("=" * 72)
print("SAMPLE CORRECT PREDICTIONS (5 examples)")
print("=" * 72)
for r in correct_samples[:5]:
    print(f"  Task: {r['sub_task']}")
    print(f"  GT: uid={r['gt_index']}, action={r['gt_action']}")
    print(f"  Pred: uid={r['pred_uid']}, action={r['pred_action']}")
    print(f"  Output: {r['raw_output'][:120]}")
    print()

print("=" * 72)
print("SAMPLE INCORRECT PREDICTIONS (5 examples)")
print("=" * 72)
for r in incorrect_samples[:5]:
    print(f"  Task: {r['sub_task']}")
    print(f"  GT: uid={r['gt_index']}, action={r['gt_action']}")
    print(f"  Pred: uid={r['pred_uid']}, action={r['pred_action']}")
    print(f"  Output: {r['raw_output'][:120]}")
    print()

print(f"\nTotal: {len(correct_samples)} correct, {len(incorrect_samples)} incorrect out of {n}")

In [ ]:
# ── Final GPU stats ──
if torch.cuda.is_available():
    print(f"Peak VRAM allocated: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
    print(f"Peak VRAM reserved:  {torch.cuda.max_memory_reserved()/1e9:.2f} GB")
    print(f"Current VRAM:        {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Disk usage
disk_stat = os.statvfs("/kaggle/working")
free_gb = disk_stat.f_bavail * disk_stat.f_frsize / 1e9
print(f"Disk free: {free_gb:.1f} GB")

print(f"\nNotebook complete. All outputs saved to /kaggle/working/prune4web_grounder/")